# 03 · Aşama 3 — Güvenilirlik farkındalıklı karar mekanizması
$$
d(x)=\begin{cases}\text{NORMAL} & p(x) > \alpha+\beta \\ \text{UNCERTAIN} & \alpha < p(x)\le \alpha+\beta \\ \text{ANOMALY} & p(x)\le\alpha\end{cases}
\qquad p(x)=\frac{1+\#\{i\in\text{kal}: s_i\ge s(x)\}}{n_{kal}+1}
$$
Bu, profesörün iki eşikli kuralının uygulanışıdır: $\tau_H,\tau_L$ normal-kalibrasyon skorlarının sıra istatistikleridir.
Tek fark sınır kuralıdır: burada ANOMALY için $s(x) > \tau_H$ kullanılır (plandaki $s(x)\ge\tau_H$ yerine); böylece eşik
kuralı p-değeri kuralıyla eşitlikler dahil **birebir eşdeğer** olur (`tests/test_conformal.py`).

**Garanti edilen** (kalibrasyon çekimine göre marjinal; normal test ve kalibrasyon örnekleri değiştirilebilir ise):
$P(\text{ANOMALY}\mid\text{normal})\le\alpha$ ve $P(\text{UNCERTAIN}\mid\text{normal})\le\beta+\tfrac{1}{n+1}$.
Garanti, **tekrar temizliği sonrası benzersiz vektörler** için geçerlidir.

**Garanti edilmeyen:** saldırılar hakkında hiçbir şey. $p(x)$ bir **olasılık değildir** ($P(\text{saldırı}\mid x)$ değil);
bu yüzden güvenilirlik burada ve Aşama 4'te **deneysel** olarak sınanır.

In [ ]:
# ---- Kurulum (her defterde aynı): Drive'ı bağla, ortak kodu içe aktar ----
import os, sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
PROJECT_DIR = os.environ.get("TRUSTIDS_PROJECT", "/content/drive/MyDrive/TrustFed-TinyIDS")
QUICK = os.environ.get("TRUSTIDS_QUICK", "0") == "1"   # hızlı mod YALNIZCA kodu test etmek içindir
# Devam modu: Colab oturumu kapanırsa, Drive'a kaydedilmiş tohumlar/adımlar atlanır ve kalan yerden devam edilir.
# Her şeyi baştan hesaplamak için: os.environ["TRUSTIDS_RESUME"] = "0"
RESUME = os.environ.get("TRUSTIDS_RESUME", "1") == "1"
sys.path.insert(0, PROJECT_DIR)

from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from trustids.config import Config
from trustids import utils

# Veri kümesi seçimi: varsayılan TON_IoT; ikinci veri kümesi için TRUSTIDS_CONFIG=edge_iiotset.json
CONFIG_NAME = os.environ.get("TRUSTIDS_CONFIG", "default.json")
CFG_PATH = Path(PROJECT_DIR) / "configs" / CONFIG_NAME
if CONFIG_NAME != "default.json" and not CFG_PATH.exists():
    raise FileNotFoundError(f"{CFG_PATH} bulunamadı: configs/ klasörünü de Drive'a yükleyin.")
cfg = Config.load(CFG_PATH) if CFG_PATH.exists() else Config(project_dir=PROJECT_DIR)
cfg.project_dir = PROJECT_DIR
if QUICK:
    cfg.train_seeds, cfg.validity_resplits, cfg.epochs = cfg.train_seeds[:2], 30, 15
    cfg.random_control_repeats, cfg.bootstrap_repeats = 10, 200
cfg.check()
ART = cfg.artifacts
FIG = Path(PROJECT_DIR) / cfg.figures_dir; FIG.mkdir(exist_ok=True)
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)
print("Proje klasörü:", PROJECT_DIR, "| hızlı mod:", QUICK, "| devam modu:", RESUME)
print("Drive'da kayıtlı ilerleme:", utils.progress(cfg))

In [ ]:
from trustids import data as D, conformal as C, metrics as MT, evaluation as E, plotting as PL
P = D.load_prepared(ART / "prepared")
SEEDS = E.available_seeds(cfg); print("skorları önbellekte olan tohumlar:", SEEDS)
S0 = E.load_scores(cfg, SEEDS[0])
GROUPS = E.family_groups(cfg)
# Aşama 1'de önceden kaydedilmiş n_cal kuralını uygula: (n_cal + 1) * alpha >= 20
rule = utils.load_json(ART / "prepared" / "prepare_report.json")["calibration_rule"]
if not rule["rule_satisfied"]:
    print("n_cal kuralı sağlanmadı -> alpha", cfg.alpha, "değerinden", rule["alpha_effective"], "değerine yükseltildi (raporlayın)")
    cfg.alpha = rule["alpha_effective"]
print("n_cal =", rule["n_cal"], "| kullanılan alpha =", cfg.alpha, "| beta =", cfg.beta)

## Eşikler, eşit skorlar ve kararlar (tohum 0)
Eşit skor (tie) içeren modellerde (Isolation Forest, ileride INT8) **rastgeleleştirilmiş p-değerleri** kullanılır; rastgelelik tohumla sabitlenir, böylece sonuçlar tekrarlanabilir.

In [ ]:
rows, DEC0 = [], {}
for model in E.MODELS:
    ev = E.eval_set(P, S0, model)
    ties = C.tie_report(ev["cal"])
    p = C.pvalues(ev["cal"], ev["scores"], seed=SEEDS[0], model=model)
    d = C.decide(p, cfg.alpha, cfg.beta)
    DEC0[model] = {"p": p, "d": d, "ev": ev}
    th = C.thresholds(ev["cal"], cfg.alpha, cfg.beta)
    rows.append({"model": model, **th, "farklı_kal_skoru": ties["distinct"], "eşitlik_oranı": ties["tie_mass"],
                 "rastgele_p": model in C.ALWAYS_RANDOMIZED_MODELS or ties["use_randomized"]})
pd.DataFrame(rows)

In [ ]:
tab = MT.per_family_table(DEC0["ae"]["ev"]["types"], DEC0["ae"]["d"], GROUPS)
tab.to_csv(FIG / "03_ae_per_family_seed0.csv", index=False)
PL.family_bars(tab, "AE + konformal üç yönlü karar, aile bazında (tohum 0)", FIG / "03_ae_family_decisions.png")
tab.round(4)

## B3 temel çizgisi: eğitim hatalarından sezgisel eşikler (μ + kσ)
Yaygın uygulama; eğitim hataları görülmemiş normal örneklerin hatalarından küçük olduğu için iyimserdir.

In [ ]:
ev = DEC0["ae"]["ev"]
h = C.heuristic_thresholds(ev["train"], cfg.heuristic_k[0], cfg.heuristic_k[1])
d_b3 = C.decide_by_thresholds(ev["scores"], h["tau_L"], h["tau_H"])
print(h)
b = ~ev["y_attack"]
print("normal ANOMALY oranı   B3 = %.4f | konformal = %.4f (bütçe alpha = %.3f)" %
      (np.mean(d_b3[b] == C.ANOMALY), np.mean(DEC0["ae"]["d"][b] == C.ANOMALY), cfg.alpha))
print("normal UNCERTAIN oranı B3 = %.4f | konformal = %.4f (bütçe beta = %.3f)" %
      (np.mean(d_b3[b] == C.UNCERTAIN), np.mean(DEC0["ae"]["d"][b] == C.UNCERTAIN), cfg.beta))

## E1 — Kural bütçelerini karşılıyor mu? (önceden kaydedilmiş karar kuralı)
Model sabit; normal-kalibrasyon ∪ normal-test R kez yeniden bölünür. Yöntem geçerli olsa bile tek tek bölmelerin
yaklaşık yarısında oran bütçeyi aşar; bu nedenle kural şudur: (a) ortalama ≤ bütçe + 2·SE,
(b) dağılım teorik Beta-Binom yasasıyla uyumlu (histogram + QQ), (c) teorik %95 yüzdeliğin üstünde kalan pay ≤ %10.

**Not:** Sabit bir havuzun rastgele yeniden bölünmesinde E1 yapı gereği sağlanır; burada esas olarak **uygulamanın
doğruluğunu** sınar. Asıl geçerlilik soruları (FL, INT8, gürültü, zaman) sonraki aşamalardadır.

In [ ]:
val_rows, KEEP = [], {}
for seed in SEEDS:
    S = E.load_scores(cfg, seed)
    for model in E.MODELS:
        pool = np.r_[S[(model, "benign_cal")], S[(model, "benign_test")]]
        n_cal = len(S[(model, "benign_cal")])
        ties = model in C.ALWAYS_RANDOMIZED_MODELS or C.tie_report(S[(model, "benign_cal")])["use_randomized"]
        note = "rastgeleleştirilmiş p" if ties else "deterministik p"
        res = C.validity_resplits(pool, n_cal, cfg.alpha, cfg.beta, cfg.validity_resplits, seed, model=model)
        a = MT.validity_summary(res["fpr"], n_cal, res["n_test"], cfg.alpha)
        u = MT.validity_summary(res["uncertain"], n_cal, res["n_test"], cfg.alpha + cfg.beta, lower=cfg.alpha)
        val_rows.append({"seed": seed, "model": model, "decision": "ANOMALY", **a, "note": note})
        val_rows.append({"seed": seed, "model": model, "decision": "UNCERTAIN", **u, "note": note})
        if seed == SEEDS[0] and model == "ae":
            KEEP = {"res": res, "n_cal": n_cal}
VAL = pd.DataFrame(val_rows); VAL.to_csv(FIG / "03_validity_E1.csv", index=False)
VAL.groupby(["model", "decision"])[["budget", "mean", "share_above_theory_q95", "rule_a_mean_ok", "rule_c_ok"]].mean().round(4)

In [ ]:
res, n_cal = KEEP["res"], KEEP["n_cal"]
bbA, _ = MT.theory_betabinom(n_cal, res["n_test"], cfg.alpha)
bbU, _ = MT.theory_betabinom(n_cal, res["n_test"], cfg.alpha + cfg.beta, lower=cfg.alpha)
PL.validity_histogram(res["fpr"], cfg.alpha, bbA, res["n_test"], "AE: yeniden bölmelerde normal ANOMALY oranı", FIG / "03_validity_anomaly_hist.png")
PL.validity_qq(res["fpr"], bbA, res["n_test"], "QQ: ANOMALY oranı - teori", FIG / "03_validity_anomaly_qq.png")
PL.validity_histogram(res["uncertain"], cfg.beta, bbU, res["n_test"], "AE: yeniden bölmelerde normal UNCERTAIN oranı", FIG / "03_validity_uncertain_hist.png");

## α ve β duyarlılığı (tohum 0, AE) — yanlış alarm ve kaçırılan saldırılara etkisi

In [ ]:
ev, p = DEC0["ae"]["ev"], DEC0["ae"]["p"]
rows = []
for a in cfg.alpha_grid:
    for b_ in cfg.beta_grid:
        d = C.decide(p, a, b_)
        ho = ev["split"] == "heldout_test"; kn = ev["split"] == "dev_test"; bn = ~ev["y_attack"]
        rows.append({"alpha": a, "beta": b_,
                     "normal_yanlis_alarm": np.mean(d[bn] == C.ANOMALY), "normal_belirsiz": np.mean(d[bn] == C.UNCERTAIN),
                     "bilinen_recall": np.mean(d[kn] == C.ANOMALY), "bilinen_kacan": np.mean(d[kn] == C.NORMAL),
                     "gorulmemis_recall": np.mean(d[ho] == C.ANOMALY), "gorulmemis_ertelenen": np.mean(d[ho] == C.UNCERTAIN),
                     "gorulmemis_kacan": np.mean(d[ho] == C.NORMAL)})
SENS = pd.DataFrame(rows); SENS.to_csv(FIG / "03_sensitivity_alpha_beta.csv", index=False); SENS.round(4)

In [ ]:
# Aşama 4 için p-değerlerini ve kararları önbelleğe al
for seed in SEEDS:
    S = E.load_scores(cfg, seed); out = {}
    for model in E.MODELS:
        ev = E.eval_set(P, S, model)
        p = C.pvalues(ev["cal"], ev["scores"], seed=seed, model=model)
        out[f"{model}__p"] = p; out[f"{model}__d"] = C.decide(p, cfg.alpha, cfg.beta)
    (ART / "decisions").mkdir(exist_ok=True)
    np.savez_compressed(ART / "decisions" / f"seed{seed}.npz", **out)
print("kararlar önbelleğe alındı, tohumlar:", SEEDS)

In [ ]:
# Colab: Drive'a yazılan tüm dosyaların kaydedildiğinden emin ol (oturumu kapatmadan önce mutlaka çalıştırın)
print("Kaydedilen ilerleme:", utils.progress(cfg))
if IN_COLAB:
    drive.flush_and_unmount()
    print("Drive kaydedildi ve ayrıldı. Sonraki defter Drive'ı yeniden bağlar.")